# Iteration, comprehensions, and generators through graph theory

Use Python's iteration model to study a finite graph, construct mathematical objects, verify
invariants, and distinguish computation from proof.

**Lecture 1 · Python Foundations I · CMOR 438 / INDE 577**


## How to use this notebook

**Estimated time:** 45 minutes of core instruction, plus 25–35 minutes of practice and extension.

**Prerequisite:** notebooks 01–04 on objects, sequences, dictionaries, and sets.

Follow **Core** during class. Predict iterator state and collection contents before running each
cell. **Practice** cells include executable mathematical checks. **Extension** sections introduce
generator functions and proof questions that may continue after class.

The graph is small enough to inspect by hand. That is deliberate: code should make the mathematics
more precise, not replace reasoning with output.


## Learning objectives

By the end of this notebook, you should be able to:

- distinguish an iterable from an iterator and explain iterator exhaustion;
- trace the protocol implemented by a `for` loop;
- use direct iteration, `enumerate`, and strict `zip` appropriately;
- accumulate results with explicit loops and control flow;
- construct lists, sets, and dictionaries with readable comprehensions;
- choose eager containers or lazy generator expressions from reuse and memory needs;
- use `sum`, `any`, and `all` with iterables;
- avoid mutation-during-iteration and silent truncation failures;
- compute vertices, degrees, edges, and isolated vertices of a finite graph; and
- explain why verifying the handshake lemma on examples is evidence, not a proof.


## Why this matters in industry

Iteration is the common interface behind in-memory collections, file streams, database cursors,
data loaders, optimization loops, and model-training batches. A pipeline can fail because an iterator
was already consumed, two streams were silently truncated by `zip`, a collection changed during
traversal, or a compact comprehension hid consequential branching.

Graph representations appear in network science, recommendation, fraud detection, molecules,
knowledge graphs, sparse matrices, and graph machine learning. Before using a graph library, we need
to understand what its algorithms consume and which graph invariants its data must satisfy.

Professional iteration code makes traversal order, stopping behavior, materialization, and mutation
explicit.


## Mathematical question and running example

Let $G=(V,E)$ be a finite simple undirected graph. Its **degree** $d(v)$ is the number of
vertices adjacent to $v$. The handshake lemma states

$$
Σ_{v ∈ V} d(v) = 2|E|.
$$

We will compute both sides for one graph, identify its isolated vertices, and validate that its
adjacency representation really describes a simple undirected graph.

The identity has a counting proof: each edge contributes one incidence to each of its two endpoints.
Our program will verify a particular representation; it will not prove the theorem for every finite
graph.


## Graph representation

Use a dictionary from each vertex to the set of its neighbors:

```text
a: {b, c}       d: {c}
b: {a, c}       e: {}
c: {a, b, d}
```

For an undirected graph, adjacency must be symmetric: if `v` is in `graph[u]`, then `u` must be in
`graph[v]`. A simple graph also has no loop from a vertex to itself. Vertex `e` is isolated.

Sets intentionally do not provide a traversal-order contract. We will call `sorted` only when a
deterministic presentation order is useful.


In [ ]:
graph = {
    "a": {"b", "c"},
    "b": {"a", "c"},
    "c": {"a", "b", "d"},
    "d": {"c"},
    "e": set(),
}

assert set(graph) == {"a", "b", "c", "d", "e"}
assert graph["e"] == set()
assert "b" in graph["a"]
assert "a" in graph["b"]


## Professional practice: mathematical and computational correctness

| Mathematician or data scientist asks | Software engineer asks |
| --- | --- |
| Does this object satisfy the graph definition? | Where are graph invariants checked? |
| Does traversal order affect the claim? | Is iteration deterministic when output must be reproducible? |
| Is the computed pattern evidence or proof? | Which tests cover empty and malformed graphs? |
| Are vertices labels or semantically meaningful entities? | Are label and identity contracts stable? |
| Could graph construction omit or duplicate edges? | Is input consumed once, copied, or mutated? |

Software checks can certify that one finite object satisfies implemented properties. General
mathematical claims still require arguments over every object covered by the hypotheses.


## Core: an iterable can produce an iterator

An **iterable** can supply its elements one at a time. Calling `iter(iterable)` returns an
**iterator**, an object holding traversal state. Calling `next(iterator)` returns the next value and
advances that state. When no value remains, `next` raises `StopIteration`.

A list, tuple, set, dictionary, string, and range are iterable. They can usually create a fresh
iterator each time. An iterator is also iterable, but `iter(iterator)` normally returns that same
stateful, one-pass object.


In [ ]:
ordered_vertices = sorted(graph)
vertex_iterator = iter(ordered_vertices)

assert iter(vertex_iterator) is vertex_iterator
assert next(vertex_iterator) == "a"
assert next(vertex_iterator) == "b"
assert list(vertex_iterator) == ["c", "d", "e"]
assert list(vertex_iterator) == []


### Practice: predict iterator state

Before executing the next cell, predict all four outputs. Pay particular attention to the second
`list(cursor)` call.

An exhausted iterator is not “empty data”; it may be a consumed view of nonempty data. When a result
must be traversed repeatedly, store an eager collection or create a fresh iterator from the original
iterable.


In [ ]:
numbers = [10, 20, 30]
cursor = iter(numbers)

print(next(cursor))
print(list(cursor))
print(list(cursor))

try:
    next(cursor)
except StopIteration:
    print("Captured StopIteration: the iterator is exhausted")

assert numbers == [10, 20, 30]


## Core: a `for` loop manages the iteration protocol

Conceptually, Python performs this process:

```text
iterator = iter(iterable)
request next value
bind value to loop target
run loop body
repeat until StopIteration
```

The loop handles `StopIteration` internally. The loop target remains bound to its final value after a
nonempty loop, so avoid relying on it as if it were scoped only inside the loop.


In [ ]:
visited_vertices = []

for vertex in ordered_vertices:
    visited_vertices.append(vertex)

assert visited_vertices == ["a", "b", "c", "d", "e"]
assert vertex == "e"
assert ordered_vertices == ["a", "b", "c", "d", "e"]


## Core: prefer direct iteration when only values matter

`for vertex in graph` directly expresses traversal over dictionary keys. Iterating a dictionary does
not produce values or key/value pairs automatically: use `.values()` or `.items()` when those are
the mathematical objects needed.

Avoid `for index in range(len(sequence))` when the index is used only to retrieve the value. Use
`enumerate` when both position and value carry meaning.


In [ ]:
degree_sum_with_loop = 0

for neighbors in graph.values():
    degree_sum_with_loop += len(neighbors)

assert degree_sum_with_loop == 8

degree_items = []
for vertex, neighbors in graph.items():
    degree_items.append((vertex, len(neighbors)))

assert degree_items == [("a", 2), ("b", 2), ("c", 3), ("d", 1), ("e", 0)]


## Core: `enumerate` pairs positions with values

`enumerate(iterable, start=0)` lazily produces `(position, value)` pairs. Use it for display ranks,
line numbers, or positions that are part of the result. It does not reorder the input.

Here we enumerate `sorted(graph)` so the printed table is deterministic. Sorting is a presentation
policy; the graph itself remains unordered with respect to vertex traversal.


In [ ]:
numbered_vertices = []

for position, vertex in enumerate(sorted(graph), start=1):
    numbered_vertices.append((position, vertex, len(graph[vertex])))

assert numbered_vertices == [
    (1, "a", 2),
    (2, "b", 2),
    (3, "c", 3),
    (4, "d", 1),
    (5, "e", 0),
]

for position, vertex, degree in numbered_vertices:
    print(f"{position}. vertex={vertex}, degree={degree}")


## Core: use strict `zip` when alignment is a contract

`zip(left, right)` lazily pairs elements position by position. By default it stops when the shortest
input is exhausted, which can silently discard unmatched data.

Python 3.10 added `zip(..., strict=True)`. Strict mode raises `ValueError` when input lengths differ.
Use it when every label must correspond to exactly one value. If unequal lengths are expected, state
and handle that policy explicitly.


In [ ]:
vertex_labels = ["a", "b", "c", "d", "e"]
vertex_degrees = [2, 2, 3, 1, 0]

aligned_pairs = list(zip(vertex_labels, vertex_degrees, strict=True))
assert aligned_pairs == [("a", 2), ("b", 2), ("c", 3), ("d", 1), ("e", 0)]

truncated_pairs = list(zip(vertex_labels, vertex_degrees[:-1]))
assert truncated_pairs == [("a", 2), ("b", 2), ("c", 3), ("d", 1)]

try:
    list(zip(vertex_labels, vertex_degrees[:-1], strict=True))
except ValueError as error:
    print(f"Captured {type(error).__name__}: {error}")


## Core: an explicit loop is often the clearest accumulation

An accumulation starts with an identity or empty result, updates it for each element, and inspects
the completed result after the loop. Examples include sums, counts, grouped records, and sets of
observed categories.

Choose an accumulator whose type matches the result. Mutation is appropriate when the accumulator is
locally owned by the computation and the source is not changed.


In [ ]:
degree_by_vertex = {}
isolated_vertices = set()

for vertex, neighbors in graph.items():
    degree_by_vertex[vertex] = len(neighbors)
    if len(neighbors) == 0:
        isolated_vertices.add(vertex)

assert degree_by_vertex == {"a": 2, "b": 2, "c": 3, "d": 1, "e": 0}
assert isolated_vertices == {"e"}
assert graph["e"] == set()


## Core: `continue`, `break`, and loop `else` control traversal

- `continue` skips the remainder of the current iteration;
- `break` stops the nearest loop immediately;
- a loop's `else` block runs only when the loop finishes without `break`.

Use these sparingly and name the state being searched for. A loop with several flags, nested breaks,
and side effects is usually a sign that the logic belongs in a function with a clear result.


In [ ]:
target_degree = 3
vertex_with_target_degree = None

for vertex, degree in degree_by_vertex.items():
    if degree != target_degree:
        continue
    vertex_with_target_degree = vertex
    break
else:
    print("No vertex has the target degree")

assert vertex_with_target_degree == "c"

for vertex, degree in degree_by_vertex.items():
    if degree == 99:
        break
else:
    impossible_degree_found = False

assert impossible_degree_found is False


## Core: do not structurally mutate a collection during iteration

Adding or removing dictionary keys or set members while iterating that same object invalidates the
traversal and usually raises `RuntimeError`. Changing values without changing dictionary size is
technically possible, but can still make reasoning difficult.

Iterate over a snapshot such as `list(mapping)` only when snapshot semantics are intended, or collect
requested changes and apply them after traversal.


In [ ]:
temporary_degrees = {"a": 2, "b": 2}

try:
    for vertex in temporary_degrees:
        temporary_degrees["c"] = 3
except RuntimeError as error:
    print(f"Captured {type(error).__name__}: {error}")

assert temporary_degrees == {"a": 2, "b": 2, "c": 3}


## Core: a comprehension constructs a collection from an iterable

Read a comprehension from left to right after locating the `for` clause:

```text
result expression  for item in iterable  if predicate
```

- `[expression for ...]` constructs a list;
- `{expression for ...}` constructs a set;
- `{key: value for ...}` constructs a dictionary.

Use a comprehension for one readable transformation or filter. Prefer an explicit loop when the body
needs several steps, error handling, logging, stateful branching, or explanatory names.


In [ ]:
degrees = [len(graph[vertex]) for vertex in sorted(graph)]
isolated_vertices = {vertex for vertex, neighbors in graph.items() if not neighbors}
degree_by_vertex = {vertex: len(neighbors) for vertex, neighbors in graph.items()}

assert degrees == [2, 2, 3, 1, 0]
assert isolated_vertices == {"e"}
assert degree_by_vertex == {"a": 2, "b": 2, "c": 3, "d": 1, "e": 0}


### Practice: translate between a loop and a comprehension

Write down the explicit loop equivalent of
`{vertex for vertex, degree in degree_by_vertex.items() if degree % 2 == 1}`.
Then run the cell and explain why the result is a set rather than a list.

The parity pattern is mathematically relevant: the handshake lemma implies every finite undirected
graph has an even number of odd-degree vertices.


In [ ]:
odd_degree_vertices = {
    vertex
    for vertex, degree in degree_by_vertex.items()
    if degree % 2 == 1
}

odd_degree_vertices_with_loop = set()
for vertex, degree in degree_by_vertex.items():
    if degree % 2 == 1:
        odd_degree_vertices_with_loop.add(vertex)

assert odd_degree_vertices == {"c", "d"}
assert odd_degree_vertices_with_loop == odd_degree_vertices
assert len(odd_degree_vertices) % 2 == 0


## Core: nested iteration can construct the edge set

The adjacency mapping mentions each undirected edge twice: once from each endpoint. Nested iteration
visits `(vertex, neighbor)` incidences. Converting each pair to a sorted tuple gives both directions
the same representation, and a set removes the duplicate.

The concise nested comprehension below is appropriate only after the traversal has been explained.
When nesting is difficult to read, keep the explicit loop.


In [ ]:
edges_with_loop = set()

for vertex, neighbors in graph.items():
    for neighbor in neighbors:
        edge = tuple(sorted((vertex, neighbor)))
        edges_with_loop.add(edge)

edges = {
    tuple(sorted((vertex, neighbor)))
    for vertex, neighbors in graph.items()
    for neighbor in neighbors
}

assert edges == {("a", "b"), ("a", "c"), ("b", "c"), ("c", "d")}
assert edges_with_loop == edges
assert len(edges) == 4


## Core: a generator expression is lazy and one-pass

Parentheses create a generator expression instead of an eager collection. It computes a value only
when a consumer requests one. This can reduce peak memory and allow streaming, but the generator is
normally consumed once.

Choose an eager list when values must be indexed, inspected repeatedly, or retained. Choose a
generator when values flow through a one-pass pipeline and retaining them is unnecessary.


In [ ]:
degree_stream = (len(graph[vertex]) for vertex in sorted(graph))

assert iter(degree_stream) is degree_stream
assert next(degree_stream) == 2
assert list(degree_stream) == [2, 3, 1, 0]
assert list(degree_stream) == []

fresh_degree_stream = (len(neighbors) for neighbors in graph.values())
assert sum(fresh_degree_stream) == 8
assert sum(fresh_degree_stream) == 0


## Core: `sum`, `any`, and `all` consume iterables

`sum` accumulates numeric values. `any` stops at the first truthy element; `all` stops at the first
falsy element. This short-circuiting can avoid unnecessary work and allows checks over lazy streams.

Two edge cases follow mathematical identities: `sum([]) == 0`, `all([]) is True`, and
`any([]) is False`. An empty collection therefore requires an explicit policy when “no violations”
and “no data” have different domain meanings.


In [ ]:
has_isolated_vertex = any(len(neighbors) == 0 for neighbors in graph.values())
has_no_self_loops = all(vertex not in neighbors for vertex, neighbors in graph.items())
all_neighbors_known = all(
    neighbor in graph
    for neighbors in graph.values()
    for neighbor in neighbors
)

assert has_isolated_vertex is True
assert has_no_self_loops is True
assert all_neighbors_known is True
assert sum([]) == 0
assert all([]) is True
assert any([]) is False


## Worked example: validate the graph and verify the handshake identity

We will check representation invariants before interpreting a theorem:

1. every neighbor is a known vertex;
2. no vertex is adjacent to itself;
3. adjacency is symmetric;
4. normalized undirected edges are unique;
5. computed degrees match adjacency sizes; and
6. the degree sum equals twice the edge count.

The checks are deterministic and independent of set traversal order.


In [ ]:
all_neighbors_known = all(
    neighbor in graph
    for neighbors in graph.values()
    for neighbor in neighbors
)
has_no_self_loops = all(vertex not in neighbors for vertex, neighbors in graph.items())
has_symmetric_adjacency = all(
    vertex in graph[neighbor]
    for vertex, neighbors in graph.items()
    for neighbor in neighbors
)

assert all_neighbors_known
assert has_no_self_loops
assert has_symmetric_adjacency


In [ ]:
degree_by_vertex = {vertex: len(neighbors) for vertex, neighbors in graph.items()}
edges = {
    tuple(sorted((vertex, neighbor)))
    for vertex, neighbors in graph.items()
    for neighbor in neighbors
}
isolated_vertices = {vertex for vertex, degree in degree_by_vertex.items() if degree == 0}

degree_sum = sum(degree_by_vertex.values())
twice_edge_count = 2 * len(edges)

assert degree_by_vertex == {"a": 2, "b": 2, "c": 3, "d": 1, "e": 0}
assert edges == {("a", "b"), ("a", "c"), ("b", "c"), ("c", "d")}
assert isolated_vertices == {"e"}
assert degree_sum == twice_edge_count == 8

print(f"degree sum = {degree_sum}")
print(f"2 × edge count = {twice_edge_count}")


### Interpretation: verification is not proof

The program establishes that **this dictionary**, under our validation rules, represents a simple
undirected graph satisfying the handshake identity. Running it on a million more graphs would increase
confidence in the implementation and might reveal counterexamples to a false conjecture.

It would still not prove the universal theorem. The proof uses a general double-counting argument:
the sum of degrees counts each edge incidence, and every undirected edge has exactly two endpoints.

Computation and proof complement one another: computation explores, tests definitions, and finds
patterns; proof explains why a statement must hold throughout its stated domain.


## Extension: a generator function suspends at `yield`

A function containing `yield` returns a generator iterator. Each `next` request resumes execution
until the next `yield`, preserving local state between requests. Generator functions are useful for
streaming parsers, combinatorial objects, graph traversals, and data pipelines.

The syntax below previews Lecture 2's function lesson. The generator yields each normalized
undirected edge once without constructing the complete edge set first.


In [ ]:
def undirected_edges(adjacency):
    """Yield each edge of a validated undirected graph once."""

    seen = set()
    for vertex, neighbors in adjacency.items():
        for neighbor in neighbors:
            edge = tuple(sorted((vertex, neighbor)))
            if edge not in seen:
                seen.add(edge)
                yield edge


edge_generator = undirected_edges(graph)

assert iter(edge_generator) is edge_generator
assert set(edge_generator) == edges
assert list(edge_generator) == []


## Extension: standard-library iterators support combinatorics

`itertools.combinations(vertices, 2)` lazily produces every unordered two-vertex subset. For five
vertices there are $C(5,2)=10$ possible edges. Subtracting actual edges identifies nonedges.

The iterator avoids materializing all candidates immediately, but collecting a set of nonedges still
uses memory proportional to the result. Laziness changes *when* values are produced; it does not make
an ultimately materialized result free.


In [ ]:
from itertools import combinations

candidate_pairs = combinations(sorted(graph), 2)
nonedges = {pair for pair in candidate_pairs if pair not in edges}

assert len(nonedges) == 6
assert ("a", "d") in nonedges
assert ("a", "b") not in nonedges
assert list(candidate_pairs) == []


## Debugging playbook for iteration problems

Inspect in this order:

1. `type(source)` — reusable iterable or one-pass iterator?
2. `iter(source) is source` — does it carry its own traversal state?
3. materialize only a small diagnostic prefix when the source could be large;
4. check lengths or use `zip(..., strict=True)` before assuming alignment;
5. verify whether a consumer already exhausted the iterator;
6. inspect traversal order and sort only when deterministic order is part of the output policy;
7. check whether the loop mutates its source;
8. expand a confusing comprehension into an explicit loop;
9. assert mathematical representation invariants before running an algorithm.

Do not “fix” an empty second result by recreating data blindly. Determine whether exhaustion was
intended, and decide which layer owns replay or materialization.


## Practice: guided cycle graph computation

The adjacency mapping below represents the four-cycle $C_4$. Derive:

- `cycle_degrees`, a dictionary of degrees;
- `cycle_edges`, a set of normalized edge tuples;
- `cycle_odd_vertices`, a set of odd-degree vertices; and
- both sides of the handshake identity.

Use comprehensions and the supplied assertions. Validate symmetry before trusting the result.


In [ ]:
cycle_graph = {
    0: {1, 3},
    1: {0, 2},
    2: {1, 3},
    3: {0, 2},
}

cycle_is_symmetric = all(
    vertex in cycle_graph[neighbor]
    for vertex, neighbors in cycle_graph.items()
    for neighbor in neighbors
)
cycle_degrees = {vertex: len(neighbors) for vertex, neighbors in cycle_graph.items()}
cycle_edges = {
    tuple(sorted((vertex, neighbor)))
    for vertex, neighbors in cycle_graph.items()
    for neighbor in neighbors
}
cycle_odd_vertices = {
    vertex for vertex, degree in cycle_degrees.items() if degree % 2 == 1
}

assert cycle_is_symmetric
assert cycle_degrees == {0: 2, 1: 2, 2: 2, 3: 2}
assert cycle_edges == {(0, 1), (0, 3), (1, 2), (2, 3)}
assert cycle_odd_vertices == set()
assert sum(cycle_degrees.values()) == 2 * len(cycle_edges) == 8


## Practice: independent path graph audit

For the proposed path graph below:

1. verify that all neighbors are known;
2. verify no self-loops and symmetric adjacency;
3. compute the degree mapping and normalized edge set;
4. identify endpoints as degree-one vertices;
5. verify the handshake identity; and
6. verify that exactly two vertices have odd degree.

The assertions state the intended result without prescribing one exact implementation.


In [ ]:
path_graph = {
    "p": {"q"},
    "q": {"p", "r"},
    "r": {"q", "s"},
    "s": {"r"},
}

path_neighbors_known = all(
    neighbor in path_graph
    for neighbors in path_graph.values()
    for neighbor in neighbors
)
path_has_no_loops = all(vertex not in neighbors for vertex, neighbors in path_graph.items())
path_is_symmetric = all(
    vertex in path_graph[neighbor]
    for vertex, neighbors in path_graph.items()
    for neighbor in neighbors
)
path_degrees = {vertex: len(neighbors) for vertex, neighbors in path_graph.items()}
path_edges = {
    tuple(sorted((vertex, neighbor)))
    for vertex, neighbors in path_graph.items()
    for neighbor in neighbors
}
path_endpoints = {vertex for vertex, degree in path_degrees.items() if degree == 1}
path_odd_vertices = {vertex for vertex, degree in path_degrees.items() if degree % 2}

assert path_neighbors_known and path_has_no_loops and path_is_symmetric
assert path_degrees == {"p": 1, "q": 2, "r": 2, "s": 1}
assert path_edges == {("p", "q"), ("q", "r"), ("r", "s")}
assert path_endpoints == {"p", "s"}
assert sum(path_degrees.values()) == 2 * len(path_edges) == 6
assert len(path_odd_vertices) == 2


## Extension: computational experiment and proof task

Design a small investigation of the conjecture “every finite undirected graph has an even number of
odd-degree vertices.” Address:

1. how candidate graphs will be represented and validated;
2. how repeated or isomorphic examples affect the evidence;
3. what empty and disconnected graphs contribute;
4. which assertions distinguish a bad graph encoding from a counterexample;
5. why exhaustive verification up to a fixed size is not a general proof; and
6. how the handshake lemma proves the conjecture.

A strong response separates software validation, finite experimental evidence, and mathematical
deduction.


## Common failure modes

| Symptom | Likely mistake | Better response |
| --- | --- | --- |
| second traversal is empty | iterator was exhausted | retain iterable or deliberately materialize |
| paired values disappear | ordinary `zip` truncated | use `strict=True` when lengths must agree |
| output changes between runs | unordered traversal treated as order | sort only at the presentation boundary |
| loop raises `RuntimeError` | dictionary/set size changed during traversal | collect changes and apply afterward |
| comprehension is unreadable | too much branching or nesting | expand into named loop stages |
| memory still grows with generator | result is ultimately materialized | stream through consumers or bound result size |
| undirected edge count doubles | both adjacency directions counted | normalize endpoints or use a seen set |
| graph algorithm gives nonsense | representation invariants were unchecked | validate known vertices, loops, and symmetry |
| millions of examples called a proof | finite evidence confused with universality | provide a general mathematical argument |

Iteration mechanics and mathematical definitions must both be correct.


## Retrieval practice

Answer without running code:

1. What is the difference between an iterable and an iterator?
2. How does a `for` loop know when to stop?
3. Why can the same list be traversed twice but a generator usually cannot?
4. When should `enumerate` replace an indexing loop?
5. What failure does `zip(..., strict=True)` expose?
6. When is an explicit loop clearer than a comprehension?
7. How do list, set, and dictionary comprehensions differ?
8. Why do `all([])` and `any([])` have different values?
9. Why does an adjacency mapping mention each undirected edge twice?
10. Why is checking the handshake identity on examples not a proof?


## Takeaway and next step

Iteration is Python's common protocol for collections and streams. Prefer direct traversal, use
strict pairing when alignment is required, keep mutation separate from traversal, and choose eager
or lazy results from how they will be consumed. Comprehensions should reveal one mathematical
transformation, not compress complicated control flow.

Our graph computations also establish a lasting course habit: validate the representation, make
invariants executable, interpret the result, and distinguish experimental evidence from proof.
Lecture 2 begins by moving repeated transformations into well-designed functions.


## Further reading

- [Python iterator types](https://docs.python.org/3.12/library/stdtypes.html#iterator-types)
- [Python `for` statement](https://docs.python.org/3.12/reference/compound_stmts.html#the-for-statement)
- [Python `enumerate`, `iter`, `next`, and `zip`](https://docs.python.org/3.12/library/functions.html)
- [Python comprehensions tutorial](https://docs.python.org/3.12/tutorial/datastructures.html#list-comprehensions)
- [Python generator expressions](https://docs.python.org/3.12/reference/expressions.html#generator-expressions)
- [Python `itertools`](https://docs.python.org/3.12/library/itertools.html)
- [PEP 618: optional length-checking for `zip`](https://peps.python.org/pep-0618/)

Use the language references to confirm mechanics. Use mathematical definitions to determine which
objects and invariants the computation must preserve.
